# Install & Imports

In [27]:
import pandas as pd    # to load dataset
import numpy as np     # for mathematic equation
from nltk.corpus import stopwords   # to get collection of stopwords
from sklearn.model_selection import train_test_split       # for splitting dataset
from tensorflow.keras.preprocessing.text import Tokenizer  # to encode text to int
from tensorflow.keras.preprocessing.sequence import pad_sequences   # to do padding or truncating
from tensorflow.keras.models import Sequential     # the model
from tensorflow.keras.layers import Embedding, LSTM, Dense # layers of the architecture
from tensorflow.keras.callbacks import ModelCheckpoint   # save model
from tensorflow.keras.models import load_model   # load saved model
import re

# Load Dataset

In [28]:
data = pd.read_csv("D:\\desktop\\6sem project\\IMDB Dataset.csv")

print(data)

                                                  review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]


# Clean Text

In [29]:
english_stops = set(stopwords.words('english'))

# Load and Clean Dataset

In [30]:
def load_dataset():
    df = pd.read_csv('D:\\desktop\\6sem project\\IMDB Dataset.csv')
    x_data = df['review']       # Reviews/Input
    y_data = df['sentiment']    # Sentiment/Output

    # PRE-PROCESS REVIEW
    x_data = x_data.replace({'<.*?>': ''}, regex = True)          # remove html tag
    x_data = x_data.replace({'[^A-Za-z]': ' '}, regex = True)     # remove non alphabet
    x_data = x_data.apply(lambda review: [w for w in review.split() if w not in english_stops])  # remove stop words
    x_data = x_data.apply(lambda review: [w.lower() for w in review])   # lower case
    
    # ENCODE SENTIMENT -> 0 & 1
    y_data = y_data.replace('positive', 1)
    y_data = y_data.replace('negative', 0)

    return x_data, y_data

x_data, y_data = load_dataset()

print('Reviews')
print(x_data, '\n')
print('Sentiment')
print(y_data)

Reviews
0        [one, reviewers, mentioned, watching, oz, epis...
1        [a, wonderful, little, production, the, filmin...
2        [i, thought, wonderful, way, spend, time, hot,...
3        [basically, family, little, boy, jake, thinks,...
4        [petter, mattei, love, time, money, visually, ...
                               ...                        
49995    [i, thought, movie, right, good, job, it, crea...
49996    [bad, plot, bad, dialogue, bad, acting, idioti...
49997    [i, catholic, taught, parochial, elementary, s...
49998    [i, going, disagree, previous, comment, side, ...
49999    [no, one, expects, star, trek, movies, high, a...
Name: review, Length: 50000, dtype: object 

Sentiment
0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 50000, dtype: int64


C:\Users\Admin\AppData\Local\Temp\ipykernel_16648\1948097605.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_data = y_data.replace('negative', 0)


# Split Dataset

In [31]:
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size = 0.2)

print('Train Set')
print(x_train, '\n')
print(x_test, '\n')
print('Test Set')
print(y_train, '\n')
print(y_test)

Train Set
35906    [cute, film, three, lively, sisters, switzerla...
37104    [perhaps, genre, plot, horrible, acting, nancy...
19261    [overall, i, call, disappointing, performance,...
38418    [this, movie, bad, i, know, begin, apparently,...
36704    [oh, dear, while, chevy, chase, gang, snl, set...
                               ...                        
24246    [the, death, performer, broadway, stage, play,...
9608     [a, must, see, film, great, dialogues, great, ...
23306    [overall, i, agree, wholly, ebert, review, in,...
7631     [friz, freleng, all, abir, r, r, one, best, sy...
2844     [i, never, seen, show, much, story, mystery, s...
Name: review, Length: 40000, dtype: object 

28982    [the, credits, come, sandy, frank, stitching, ...
4591     [i, enjoyed, series, felt, whole, thing, let, ...
25795    [overall, pretty, bad, film, but, pathmark, to...
21431    [rented, one, accidentally, behind, movie, box...
8646     [how, many, corrupt, cops, without, one, slipp...
 

In [32]:
def get_max_length():
    review_length = []
    for review in x_train:
        review_length.append(len(review))

    return int(np.ceil(np.mean(review_length)))

# Tokenize and Pad/Truncate Reviews

In [33]:
# ENCODE REVIEW
token = Tokenizer(lower=False)    # no need lower, because already lowered the data in load_data()
token.fit_on_texts(x_train)
x_train = token.texts_to_sequences(x_train)
x_test = token.texts_to_sequences(x_test)

max_length = get_max_length()

x_train = pad_sequences(x_train, maxlen=max_length, padding='post', truncating='post')
x_test = pad_sequences(x_test, maxlen=max_length, padding='post', truncating='post')

total_words = len(token.word_index) + 1   # add 1 because of 0 padding

print('Encoded X Train\n', x_train, '\n')
print('Encoded X Test\n', x_test, '\n')
print('Maximum review length: ', max_length)

Encoded X Train
 [[  927     4   184 ...  7869   297  8054]
 [  293   423    44 ...     0     0     0]
 [  347     1   543 ...     0     0     0]
 ...
 [  347     1   910 ...     0     0     0]
 [26701 22828   201 ...     0     0     0]
 [    1    41    38 ...     0     0     0]] 

Encoded X Test
 [[   2  818  121 ...    0    0    0]
 [   1  425  110 ...    0    0    0]
 [ 347   91   19 ...    0    0    0]
 ...
 [ 146    1 1059 ...    1   47    1]
 [8052 8580    4 ...    0    0    0]
 [   8  239    4 ...    0    0    0]] 

Maximum review length:  130


In [34]:
# Build Architecture/Model

# Evaluate Model

In [35]:
# ARCHITECTURE
EMBED_DIM = 32
LSTM_OUT = 64

model = Sequential()
model.add(Embedding(total_words, EMBED_DIM, input_length = max_length))
model.add(LSTM(LSTM_OUT))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

print(model.summary())


c:\Users\Admin\.conda\envs\tf-python31310\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


# Training

In [36]:
checkpoint = ModelCheckpoint(
    'models/LSTM.h5',
    monitor='accuracy',
    save_best_only=True,
    verbose=1
)

model.fit(x_train, y_train, batch_size = 128, epochs = 5, callbacks=[checkpoint])

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.5532 - loss: 0.6705
Epoch 1: accuracy improved from None to 0.58725, saving model to models/LSTM.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 59s 178ms/step - accuracy: 0.5872 - loss: 0.6618
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 0.6882 - loss: 0.6055
Epoch 2: accuracy improved from 0.58725 to 0.73013, saving model to models/LSTM.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 64s 205ms/step - accuracy: 0.7301 - loss: 0.5681
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.6814 - loss: 0.5897
Epoch 3: accuracy did not improve from 0.73013
313/313 ━━━━━━━━━━━━━━━━━━━━ 62s 198ms/step - accuracy: 0.6672 - loss: 0.6065
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.8335 - loss: 0.4396
Epoch 4: accuracy improved from 0.73013 to 0.84930, saving model to models/LSTM.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 74s 170ms/step - accuracy: 0.8493 - loss: 0.3893
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step - accuracy: 0.9033 - loss: 0.2643
Epoch 5: accuracy improved from 0.84930 to 0.90475, saving model to models/LSTM.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 59s 188ms/step - accuracy: 0.9047 - loss: 0.2561


# Testing

In [37]:
y_pred = (model.predict(x_test, batch_size=128) > 0.5).astype("int32")

true = 0

for i, y in enumerate(y_test):
    if y == y_pred[i]:
        true += 1

print("Correct Prediction:", true)
print("Wrong Prediction:", len(y_pred) - true)
print("Accuracy:", true/len(y_pred)*100)

79/79 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step
Correct Prediction: 8698
Wrong Prediction: 1302
Accuracy: 86.98


# Load Model + Predict New Text

In [48]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(x_data)

In [50]:
model = load_model("models/LSTM.h5")

import pickle

with open("config.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [71]:
review = str(input('Movie Review: '))

In [72]:
# Pre-process input
regex = re.compile(r'[^a-zA-Z\s]')
review = regex.sub('', review)
print('Cleaned: ', review)

words = review.split(' ')
filtered = [w for w in words if w not in english_stops]
filtered = ' '.join(filtered)
filtered = [filtered.lower()]

print('Filtered: ', filtered)

Cleaned:  this is a comedy movie and i watch that movie many times i recommend this movie to my friends
Filtered:  ['comedy movie watch movie many times recommend movie friends']


In [73]:
tokenize_words = token.texts_to_sequences(filtered)
tokenize_words = pad_sequences(tokenize_words, maxlen=max_length, padding='post', truncating='post')
print(tokenize_words)

[[109   3  33   3  37 117 284   3 258   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0]]


In [74]:
result = loaded_model.predict(tokenize_words)
print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
[[0.81486]]


In [75]:
if result >= 0.7:
    print('positive')
else:
    print('negative')

positive
